In [ ]:

# KAGGLE DEFAULT WORKFLOW VARIABLES
PREP_ONLY = True
START_TRAINING = False
RUN_SMOKE = False

import os
import sys
import json
import hashlib
import zipfile
import subprocess
from pathlib import Path

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192): h.update(chunk)
    return h.hexdigest()


In [ ]:

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")
    if START_TRAINING:
        raise RuntimeError("No CUDA GPU available, refusing to start full training.")


In [ ]:

print("Discovering bundle...")
# Known canonical SHA for the V2 pretrain bundle
EXPECTED_SHA = "678b3a6e2b5a6ccb0b4db939303f4f97743e0ef41f0d51273028e240c3a021aa"

target_bundle = None
input_dir = Path('/kaggle/input')
if input_dir.exists():
    for p in input_dir.rglob('*.zip'):
        print(f"Checking {p}...")
        try:
            if sha256_file(p) == EXPECTED_SHA:
                target_bundle = p
                break
        except Exception:
            pass

if not target_bundle:
    print("Checking if already extracted (Kaggle auto-extract)...")
    for p in input_dir.rglob('V2_PORTABLE_EXPECTED_HASHES.json'):
        print("Found extracted manifest, using directory as bundle root.")
        target_bundle = p.parent
        break

if not target_bundle:
    raise RuntimeError(f"Could not find valid bundle matching SHA {EXPECTED_SHA}")

print(f"Found valid bundle at: {target_bundle}")

WORKING_DIR = Path('/kaggle/working/roadwatch_v2')
if WORKING_DIR.exists():
    import shutil
    shutil.rmtree(WORKING_DIR)
WORKING_DIR.mkdir(parents=True)

if target_bundle.is_file():
    print("Extracting ZIP...")
    with zipfile.ZipFile(target_bundle, 'r') as zf:
        zf.extractall(WORKING_DIR)
else:
    print("Copying extracted files...")
    import shutil
    shutil.copytree(target_bundle, WORKING_DIR, dirs_exist_ok=True)

sys.path.insert(0, str(WORKING_DIR))


In [ ]:

print("Running Preflight...")
import subprocess
try:
    subprocess.check_call(["python", "training/audit_v2_readiness.py"], cwd=str(WORKING_DIR))
    print("KAGGLE_PREFLIGHT = PASS")
except subprocess.CalledProcessError:
    print("KAGGLE_PREFLIGHT = FAIL")
    if START_TRAINING or RUN_SMOKE:
        raise RuntimeError("Preflight failed. Refusing to train.")


In [ ]:

if RUN_SMOKE and not START_TRAINING:
    print("Running ONE-EPOCH SMOKE...")
    try:
        subprocess.check_call([
            "python", "-c",
            "import os; from training.v2_pretrain_portable_runner import main; os.environ['V2_SMOKE_TEST']='1'; main()"
        ], cwd=str(WORKING_DIR))
        print("SMOKE_PASS = True")
    except Exception as e:
        print(f"Smoke test failed: {e}")
        raise


In [ ]:

if START_TRAINING:
    print("Starting full training...")
    try:
        subprocess.check_call(["python", "-m", "training.v2_pretrain_portable_runner"], cwd=str(WORKING_DIR))
    except Exception as e:
        print(f"Training failed: {e}")
        raise
else:
    print("TRAINING IS DISABLED BY DEFAULT. Set START_TRAINING = True to proceed.")
